# 사람 배치 기록의 상태·규칙 근거 점검

왜: 입력 순서와 최종 배치를 연결한 것만으로 과거 게임을 재현할 수는 없습니다.
이번에는 **같은 작은 Train 표본에 어떤 상태·규칙 근거가 남아 있는지** 확인합니다.

[실험 명세](../../../ai-harness/ml-experiments/experiments/human-placement-context-evidence.md) · [연결 계약](../../../docs/ai-ml/contracts/human-placement-action-link.md)

게임 엔진·replay·모델 학습은 실행하지 않습니다. 기본 모두 실행은 가상 Smoke만 수행합니다.
구성: 공통 정의 → 작은 예제/검사 → 가상 Smoke → 선택적인 실제 세 표본 → Full 차단 → 해석.


In [ ]:
import copy
import hashlib
import json
import os
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

EXPERIMENT_ID = "human-placement-context-evidence"
ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_context_evidence.ipynb"
LINK_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_action_link_readiness.ipynb"
LINK_DEFS_SHA = "4814ffefdc5aa929106ed5676afe3789288633bab57c7c8dc9bfa54c0d839ce3"


def link_definitions():
    return [{"id": c["id"], "source": "".join(c["source"])}
            for c in json.loads(LINK_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
            if c["cell_type"] == "code" and "definitions" in c["metadata"].get("tags", [])]


def definition_sha(cells):
    return hashlib.sha256((json.dumps(cells, ensure_ascii=False, sort_keys=True,
                                     allow_nan=False, indent=2) + "\n").encode()).hexdigest()


definitions = link_definitions()
assert len(definitions) == 5 and definition_sha(definitions) == LINK_DEFS_SHA
link = {"__name__": "context_link_shared"}
for cell in definitions:
    exec(compile(cell["source"], str(LINK_NOTEBOOK), "exec"), link)
for name in ("require", "finite", "integer", "json_bytes", "sha256_file", "normalize_event"):
    globals()[name] = link[name]
link["dataset"]["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID


## 1. 같은 이름의 필드라도 의미를 나눕니다

| 확인할 값 | 말할 수 있는 것 | 말할 수 없는 것 |
| --- | --- | --- |
| ruleset/stage/build 표기 | 명시적인 marker의 존재 | 그 marker에 대응하는 과거 빌드 검증 |
| 입력 createdAt 차이 | 기록된 처리 시도 간격 | 인지 반응시간·tick의 ms 환산 |
| 이전 boardAfter = 다음 boardBefore | 두 저장 snapshot의 셀 일치 | 모든 중간 입력의 보드가 같았음 |
| 같은 rotation의 상대 셀 일치 | 저장 표현끼리의 일관성 | 실제 회전 규칙·도달 가능성 검증 |

결과 보드·최종 배치·마지막 입력 여부는 **사후 진단 전용**입니다. 학습 feature로 넘기지 않습니다.


In [ ]:
def board_ok(board):
    return isinstance(board, list) and len(board) == 20 and all(
        isinstance(row, list) and len(row) == 10 and all(
            x is None or isinstance(x, str) and bool(x.strip()) for x in row) for row in board)


def epoch_ms(value):
    """표기된 epoch의 수치만 변환한다. 서로 같은 clock이라는 근거는 만들지 않는다."""
    if not isinstance(value, str):
        return None
    try:
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
        return round(parsed.timestamp() * 1000, 3) if parsed.tzinfo is not None else None
    except (ValueError, OverflowError):
        return None


def relative_cells(piece):
    link["physical_key"](piece)
    return sorted((c["row"] - piece["row"], c["col"] - piece["col"]) for c in piece["cells"])


def segment_evidence(line, raw):
    """검증된 로그를 요약한다. 필드 존재와 검증/실행 성공을 혼동하지 않는다."""
    _, event, payload, trust = normalize_event(raw)
    require(event["eventType"] == "placement_decision" and payload.get("source") == "human",
            "HUMAN_PLACEMENT_REQUIRED")
    require(board_ok(payload.get("boardBefore")), "BOARD_INVALID")
    active, selected = payload.get("activePiece"), payload.get("selectedPlacement")
    link["physical_key"](active)
    link["physical_key"](selected, selected=True)
    sequence = payload.get("inputSequence")
    require(isinstance(sequence, list) and len(sequence) >= 2 and all(
        isinstance(i, dict) and isinstance(i.get("action"), str)
        and i["action"] in link["dataset"]["shared"]["ACTIONS"]
        and all(integer(i.get(k)) for k in ("sequence", "tick", "pieceIndex"))
        and finite(i.get("createdAt")) and i["createdAt"] >= 0 for i in sequence), "INPUT_INVALID")
    times = [i["createdAt"] for i in sequence]
    intervals = [b - a for a, b in zip(times, times[1:])]
    observation = payload.get("observation")
    stage = observation.get("stage") if isinstance(observation, dict) else None
    rules = payload.get("gameplayRulesVersion")
    marker_names = ("buildId", "buildVersion", "gitSha", "commitSha", "clientBuildVersion", "serverBuildVersion")
    markers = [(k, value[k]) for value in (raw, payload) for k in marker_names if k in value]
    valid_markers = sum(isinstance(value, str) and bool(value.strip()) for _, value in markers)
    same_rotation = active["pieceType"] == selected["pieceType"] and active["rotation"] == selected["rotation"]
    offsets = ("same_rotation_match" if relative_cells(active) == relative_cells(selected)
               else "same_rotation_mismatch") if same_rotation else "different_rotation_not_compared"
    declared_time = epoch_ms(raw.get("occurred_at"))
    return {"sourceLine": line, "pieceIndex": event["pieceIndex"], "pieceType": active["pieceType"],
            "schemaVersion": event["schemaVersion"], "trust": trust, "boardRows": 20,
            "rulesetDeclared": isinstance(rules, str) and bool(rules.strip()),
            "stageDeclared": isinstance(stage, str) and bool(stage.strip()),
            "buildMarkerPresent": valid_markers > 0, "invalidBuildMarkerCount": len(markers) - valid_markers,
            "buildBindingVerified": False,
            "inputCount": len(sequence), "recordedSpanMs": times[-1] - times[0],
            "zeroIntervals": sum(t == 0 for t in intervals), "timeReversals": sum(t < 0 for t in intervals),
            "sequenceGaps": sum(b["sequence"] != a["sequence"] + 1 for a, b in zip(sequence, sequence[1:])),
            "beforeAfterStateFields": sum(isinstance(i.get("activePieceBefore"), dict)
                                         and isinstance(i.get("activePieceAfter"), dict) for i in sequence),
            "outcomeFields": sum(type(i.get("applied")) is bool for i in sequence),
            "lastRecordedActionIsHardDrop": sequence[-1]["action"] == "hard_drop",
            "numericEventMinusLastInputMs": None if declared_time is None else round(declared_time - times[-1], 3),
            "eventTickEqualsLastInputTick": payload.get("tick") == sequence[-1]["tick"],
            "storedOffsets": offsets, "replayExecuted": False}


def audit_context(records, scope):
    require(scope in ("synthetic", "train_sample"), "FULL_NOT_AUTHORIZED")
    rows, rejected, normalized = [], [], {}
    for line, raw in sorted(records.items()):
        try:
            row = segment_evidence(line, raw)
            rows.append(row)
            _, event, payload, _ = normalize_event(raw)
            normalized[line] = (event, payload)
        except ValueError as error:
            rejected.append({"sourceLine": line, "reason": str(error)})
    boundaries = []
    for left, right in zip(sorted(normalized), sorted(normalized)[1:]):
        a, ap = normalized[left]
        b, bp = normalized[right]
        consecutive = (a["gameId"] == b["gameId"] and a["playerId"] == b["playerId"]
                       and b["pieceIndex"] == a["pieceIndex"] + 1)
        if not consecutive:
            boundaries.append({"leftLine": left, "rightLine": right, "status": "not_consecutive_same_actor",
                               "changedCells": None})
            continue
        before, after = ap.get("boardAfter"), bp["boardBefore"]
        if not board_ok(before):
            boundaries.append({"leftLine": left, "rightLine": right, "status": "left_board_after_unverified",
                               "changedCells": None})
            continue
        changed = sum(x != y for ar, br in zip(before, after) for x, y in zip(ar, br))
        boundaries.append({"leftLine": left, "rightLine": right,
                           "status": "stored_snapshots_equal" if changed == 0 else "stored_snapshots_differ",
                           "changedCells": changed})
    return {"segments": rows, "boundaries": boundaries, "rejected": rejected,
            "counts": {"requestedSegments": len(records), "inspectedSegments": len(rows),
                       "quarantinedSegments": len(rejected), "boundaryCandidates": len(boundaries),
                       "comparedBoundaries": sum(b["changedCells"] is not None for b in boundaries)}}


In [ ]:
def synthetic_records():
    records, _, _ = link["synthetic_case"]()
    first = records[1]
    first["payload"]["boardAfter"] = copy.deepcopy(first["payload"]["boardBefore"])
    first["occurred_at"] = "1970-01-01T00:00:00.250+00:00"
    second = copy.deepcopy(first)
    second["event_id"] = "synthetic-second"
    second["piece_index"] = 2
    second["occurred_at"] = "1970-01-01T00:00:00.750+00:00"
    for item in second["payload"]["inputSequence"]:
        item["pieceIndex"] = 2
        item["createdAt"] += 500
    return {1: first, 2: second}


def report_context(records, scope, provenance):
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID,
            "status": "context_evidence_audit_complete", "runMode": "smoke", "auditScope": scope,
            "fullCohortCovered": False, "policyDecisionEligible": False, "canonicalResultEligible": False,
            "productionIntegration": False, "modelFitCount": 0, "replayExecuted": False,
            "replayReadiness": "not_established", "stateConditionedTrainingReady": False,
            "ingestionClockUsed": False, "testFileRead": False, "validationFileRead": False,
            "sourceUnchanged": True, "sourceNotebookSha256": sha256_file(SOURCE_NOTEBOOK),
            "linkDefinitionSha256": LINK_DEFS_SHA, "provenance": provenance, **audit_context(records, scope)}


def save_context(report, output):
    if output is None:
        return
    destination = Path(output)
    if not destination.is_absolute():
        destination = ROOT / destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    path = link["checked_paths"](ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(destination), "smoke")[1]
    link["publish_report"](report, path)


def run_context_sample(output=None):
    """기존 연결 검사를 재사용하고 같은 raw 세 줄의 추가 근거만 점검한다."""
    require(definition_sha(link_definitions()) == LINK_DEFS_SHA, "LINK_DEFINITIONS_CHANGED")
    own_sha, link_sha = sha256_file(SOURCE_NOTEBOOK), sha256_file(LINK_NOTEBOOK)
    linked = link["run_train_sample"]()
    require(linked["counts"]["linkedSegments"] == 3 and not linked["rejected"], "LINK_SAMPLE_NOT_VERIFIED")
    source = link["repo_input"](os.environ.get("HUMAN_PLACEMENT_LINK_RAW", link["dataset"]["DEFAULT_SOURCE"]))
    require(sha256_file(source) == link["RAW_SHA"], "SOURCE_SHA_MISMATCH")
    wanted = set(link["SAMPLE_LINES"])
    records = {}
    with source.open("rb") as handle:
        for line, data in enumerate(handle, 1):
            if line in wanted:
                records[line] = json.loads(data, parse_constant=link["reject_constant"],
                                           parse_float=link["finite_float"])
            if line == max(wanted):
                break
    require(set(records) == wanted, "SOURCE_LINES_MISSING")
    report = report_context(records, "train_sample", {"rawSha256": link["RAW_SHA"],
        "selectedSourceLines": sorted(wanted), "rawRecordsParsed": len(records),
        "linkedPairs": linked["counts"]["linkedPairs"], "priorInputChecksums": linked["inputs"]})
    require(sha256_file(source) == link["RAW_SHA"] and sha256_file(SOURCE_NOTEBOOK) == own_sha
            and sha256_file(LINK_NOTEBOOK) == link_sha, "INPUT_CHANGED")
    save_context(report, output)
    return report


## 2. 작은 예제와 내부 검사
가상 기록 두 개에서 저장된 이전 결과 보드와 다음 입력 보드를 같게 만들었습니다.
이를 보고 “snapshot 두 개가 같다”고만 말해야 합니다. 중간 조작 상태가 복원됐다는 뜻은 아닙니다.

같은 회전의 상대 셀이 일치해도 실제 core의 조각 정의·충돌 규칙을 검증한 것이 아닙니다.


In [ ]:
def self_checks():
    import tempfile
    checks = []
    records = synthetic_records()
    original = json_bytes(records)
    audit = audit_context(records, "synthetic")
    assert audit["boundaries"][0]["status"] == "stored_snapshots_equal"
    assert audit["segments"][0]["numericEventMinusLastInputMs"] == 0
    assert audit["segments"][0]["storedOffsets"] == "same_rotation_match"
    assert not audit["segments"][0]["buildBindingVerified"]
    assert json_bytes(records) == original
    changed = copy.deepcopy(records); changed[2]["payload"]["boardBefore"][0][0] = "bar"
    assert audit_context(changed, "synthetic")["boundaries"][0]["changedCells"] == 1
    changed[2]["player_id"] = "different"
    assert audit_context(changed, "synthetic")["boundaries"][0]["status"] == "not_consecutive_same_actor"
    checks.append("snapshot_identity_difference_not_replay")

    changed = copy.deepcopy(records)
    changed[1]["payload"]["selectedPlacement"]["col"] += 1
    assert audit_context(changed, "synthetic")["segments"][0]["storedOffsets"] == "same_rotation_mismatch"
    changed[1]["payload"]["selectedPlacement"]["rotation"] = 1
    assert audit_context(changed, "synthetic")["segments"][0]["storedOffsets"] == "different_rotation_not_compared"
    changed = copy.deepcopy(records); changed[1]["payload"]["buildId"] = []
    assert audit_context(changed, "synthetic")["segments"][0]["invalidBuildMarkerCount"] == 1
    changed[1]["payload"]["buildId"] = "declared-only"
    row = audit_context(changed, "synthetic")["segments"][0]
    assert row["buildMarkerPresent"] and not row["buildBindingVerified"]
    checks.append("offsets_and_markers_do_not_establish_build_binding")

    assert epoch_ms("1970-01-01T00:00:00.250+00:00") == 250
    assert epoch_ms("1970-01-01T00:00:00") is None
    assert epoch_ms("invalid") is None
    changed = copy.deepcopy(records); changed[1]["payload"]["inputSequence"][0]["createdAt"] = True
    assert audit_context(changed, "synthetic")["rejected"][0]["reason"] == "INPUT_INVALID"
    changed = copy.deepcopy(records); changed[1]["payload"]["boardBefore"] = [[None]]
    assert audit_context(changed, "synthetic")["rejected"][0]["reason"] == "BOARD_INVALID"
    report = report_context(records, "synthetic", {"kind": "synthetic_only"})
    assert not report["ingestionClockUsed"] and not report["replayExecuted"]
    assert not report["stateConditionedTrainingReady"] and not report["policyDecisionEligible"]
    assert json_bytes(report) == json_bytes(report_context(records, "synthetic", {"kind": "synthetic_only"}))
    checks.append("clock_malformed_input_and_determinism")
    with tempfile.TemporaryDirectory() as folder:
        leaf = Path(folder) / "report.json"
        save_context(report, leaf)
        for target in (leaf, Path(folder) / "child" / ".." / "report.json"):
            try: save_context(report, target)
            except ValueError: pass
            else: raise AssertionError("EXISTING_OUTPUT_OVERWRITTEN")
    checks.append("exclusive_output")
    return pd.DataFrame({"check": checks, "result": "passed"})


display(self_checks())


In [ ]:
def show_context(report):
    """보존한 근거 표·분포만 표시한다. 추론이나 replay를 실행하지 않는다."""
    print(report["auditScope"], "|", report["status"], "| fit=0, replay=false")
    display(pd.DataFrame([report["counts"]]))
    frame = pd.DataFrame(report["segments"])
    display(frame)
    display(pd.DataFrame(report["boundaries"]))
    display(pd.DataFrame(report["rejected"], columns=["sourceLine", "reason"]))
    if frame.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    columns = ["rulesetDeclared", "stageDeclared", "buildMarkerPresent"]
    frame[columns].sum().plot.bar(ax=axes[0], rot=20)
    axes[0].set(ylabel="sample segment count", title="Marker presence is not verification")
    axes[0].set_ylim(0, len(frame) + 0.5)
    axes[1].bar(frame["sourceLine"].astype(str), frame["recordedSpanMs"])
    axes[1].set(xlabel="source line", ylabel="recorded attempt span (ms)", title="Not reaction time")
    fig.suptitle(report["auditScope"])
    plt.tight_layout(); plt.show(); plt.close(fig)
    print("Equal snapshots/offsets do not prove replay. Missing evidence is not corrupt data.")


## 3. 가상 Smoke — 설정 → 실행 → 결과
기본 모두 실행은 이 가상 예제만 처리합니다. 실제 dataset/report/raw를 열지 않습니다.


In [ ]:
SMOKE_OUTPUT = os.environ.get("HUMAN_CONTEXT_SMOKE_REPORT") or None


In [ ]:
SMOKE_REPORT = report_context(synthetic_records(), "synthetic", {"kind": "synthetic_only"})
save_context(SMOKE_REPORT, SMOKE_OUTPUT)


In [ ]:
show_context(SMOKE_REPORT)


## 4. 작은 Train 표본 — 설정 → 실행 → 결과
실제로 확인할 때만 `RUN_CONTEXT_SAMPLE=True`로 변경한 뒤 설정 → 실행 → 결과 셀을 실행하세요.
이전 연결 점검의 세 sourceLine만 사용합니다. `SAMPLE_OUTPUT=None`이면 메모리 결과만 표시합니다.
저장하려면 새 JSON 경로를 지정하세요. 기존 결과는 덮어쓰지 않습니다.

이 점검은 과거 게임의 입력 재생이나 전체 데이터의 학습 준비 판정이 아닙니다.


In [ ]:
RUN_CONTEXT_SAMPLE = False
SAMPLE_OUTPUT = os.environ.get("HUMAN_CONTEXT_SAMPLE_REPORT") or None


In [ ]:
CONTEXT_SAMPLE_REPORT = None
if RUN_CONTEXT_SAMPLE:
    CONTEXT_SAMPLE_REPORT = run_context_sample(SAMPLE_OUTPUT)
else:
    print("실제 표본은 실행하지 않았습니다. RUN_CONTEXT_SAMPLE=True로 별도 선택하세요.")


In [ ]:
if CONTEXT_SAMPLE_REPORT is not None:
    show_context(CONTEXT_SAMPLE_REPORT)
else:
    print("실제 표본 결과 없음. 위 표/그림은 가상 Smoke입니다.")


## 5. Full·실제 replay — 이번 범위에서 실행 불가
전체 cohort·역사 build의 규칙 replay·dataset 생성·학습은 구현하지 않았습니다.
Full 여부나 실제 상태 복원은 별도 범위와 근거를 정해야 합니다.


In [ ]:
RUN_FULL = False
require(not RUN_FULL, "FULL_AND_REPLAY_REQUIRE_SEPARATE_SPEC")


## 6. 해석 순서
1. 실제 표본인지 가상 자료인지 먼저 확인합니다.
2. 규칙·관측 단계·build marker의 존재와 잘못된 값 수를 봅니다. marker가 있어도 binding 검증과 다릅니다.
3. 기록된 입력 시도 순서/간격과 전후 상태/outcome 필드가 얼마나 있는지 봅니다.
4. 경계 보드가 같으면 두 snapshot의 일치, 다르면 차이 셀 수만 해석합니다. 중간 사건의 원인은 단정하지 않습니다.
5. 같은 회전의 상대 셀이 다르면 저장 표현을 추가 조사합니다. 다른 회전은 이번 비교 대상이 아닙니다.
6. 결론이 미확인이어도 로그를 버리거나 새 수집을 요구하지 않습니다. 가능한 오프라인 과제와 실제 controller의 미확인 부분을 구분합니다.

이번 결과는 기존 raw의 한정된 세 표본에 대한 근거입니다. 현재 엔진으로 과거 플레이를 재현했다거나
사람의 의도·입력 성공·상황 조건부 정책을 학습했다는 뜻이 아닙니다. 다음은 사용자 실행 결과의 해석이며
전체 검사·replay·모델 학습·사이트 적용을 자동 시작하지 않습니다.
